In [27]:
#cargamos las tres novelas y aplicamos segmentacion y tokenizacion

from pathlib import Path
import re
import pandas as pd
import random

CSV_PATH = Path("gutenberg_novels_dataset.csv")

df = pd.read_csv(CSV_PATH)

#revisamos los libros disponibles
print("Libros encontrados en el dataset")
print(df[["title", "author"]])

#seleccionamos las tres novelas del laboratorio
novelas = df[df["title"].isin([
    "Pride and Prejudice",
    "Frankenstein",
    "Dracula"
])].copy()

if len(novelas) != 3:
    raise ValueError("No se encontraron exactamente las tres novelas esperadas")

#tokenizamos cada novela por separado
patron_palabras = re.compile(
    r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+(?:[-'][A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+)*|\d+(?:[.,]\d+)*",
    re.UNICODE
)

#procesamos cada novela y guardamos los resultados en un diccionario
novelas_procesadas = {}

for _, fila in novelas.iterrows():
    texto = str(fila["text"])
    texto = re.sub(r"\s+", " ", texto).strip()

    #hacemos la segmentacion en oraciones
    oraciones_raw = re.split(r"(?<=[.!?])\s+", texto)

    #tokenizamos cada oracion sin quitar stopwords ni lematizar
    oraciones_tokenizadas = [
        patron_palabras.findall(oracion)
        for oracion in oraciones_raw
    ]

    oraciones_tokenizadas = [
        oracion
        for oracion in oraciones_tokenizadas
        if len(oracion) > 0
    ]

    novelas_procesadas[fila["title"]] = {
        "autor": fila["author"],
        "texto": texto,
        "oraciones": oraciones_tokenizadas
    }

    print(f"\nLibro {fila['title']}")
    print(f"Autor {fila['author']}")
    print(f"Total de oraciones {len(oraciones_tokenizadas):,}")
    print(f"Total de tokens {sum(len(oracion) for oracion in oraciones_tokenizadas):,}")

Libros encontrados en el dataset
                 title        author
0  Pride and Prejudice   Jane Austen
1         Frankenstein  Mary Shelley
2              Dracula   Bram Stoker

Libro Pride and Prejudice
Autor Jane Austen
Total de oraciones 5,943
Total de tokens 128,366

Libro Frankenstein
Autor Mary Shelley
Total de oraciones 3,122
Total de tokens 75,303

Libro Dracula
Autor Bram Stoker
Total de oraciones 7,839
Total de tokens 162,771


In [28]:
#seleccionamos una muestra aleatoria de 100 oraciones por novela

SEED = 42
random.seed(SEED)

TAMANO_MUESTRA = 100

for titulo, datos in novelas_procesadas.items():
    if len(datos["oraciones"]) < TAMANO_MUESTRA:
        raise ValueError(f"{titulo} no tiene suficientes oraciones para crear la muestra")

    datos["muestra"] = random.sample(
        datos["oraciones"],
        TAMANO_MUESTRA
    )

    datos["tokens_muestra"] = sum(
        len(oracion)
        for oracion in datos["muestra"]
    )
    
    #mostramos un resumen de la muestra

    print(f"\nMuestra de {titulo}")
    print(f"Oraciones seleccionadas {len(datos['muestra']):,}")
    print(f"Tokens en la muestra {datos['tokens_muestra']:,}")
    print("Ejemplo de oracion de la muestra")
    print(datos["muestra"][0])


Muestra de Pride and Prejudice
Oraciones seleccionadas 100
Tokens en la muestra 1,980
Ejemplo de oracion de la muestra
['After', 'this', 'day', 'Jane', 'said', 'no', 'more', 'of', 'her', 'indifference']

Muestra de Frankenstein
Oraciones seleccionadas 100
Tokens en la muestra 2,309
Ejemplo de oracion de la muestra
['I', 'was', 'partly', 'urged', 'by', 'curiosity', 'and', 'compassion', 'confirmed', 'my', 'resolution']

Muestra de Dracula
Oraciones seleccionadas 100
Tokens en la muestra 1,731
Ejemplo de oracion de la muestra
['I', 'shall', 'wire', 'to', 'my', 'people', 'to', 'have', 'horses', 'and', 'carriages', 'where', 'they', 'will', 'be', 'most', 'convenient', 'Look', 'here', 'old', 'fellow', 'said', 'Morris', 'it', 'is', 'a', 'capital', 'idea', 'to', 'have', 'all', 'ready', 'in', 'case', 'we', 'want', 'to', 'go', 'horsebacking', 'but', 'don', 't', 'you', 'think', 'that', 'one', 'of', 'your', 'snappy', 'carriages', 'with', 'its', 'heraldic', 'adornments', 'in', 'a', 'byway', 'of', '

In [29]:
#construimos la tabla comparativa de libros completos y muestras

tabla_muestras = []

for titulo, datos in novelas_procesadas.items():
    tabla_muestras.append({
        "libro": titulo,
        "oraciones_totales": len(datos["oraciones"]),
        "tokens_totales": sum(len(oracion) for oracion in datos["oraciones"]),
        "oraciones_muestra": len(datos["muestra"]),
        "tokens_muestra": datos["tokens_muestra"]
    })

tabla_muestras = pd.DataFrame(tabla_muestras)

display(tabla_muestras)

print("\nResumen de la preparacion de la muestra")

for _, fila in tabla_muestras.iterrows():
    print(
        f"{fila['libro']} | "
        f"oraciones totales {fila['oraciones_totales']:,} | "
        f"tokens totales {fila['tokens_totales']:,} | "
        f"oraciones muestra {fila['oraciones_muestra']:,} | "
        f"tokens muestra {fila['tokens_muestra']:,}"
    )

,libro,oraciones_totales,tokens_totales,oraciones_muestra,tokens_muestra
0,Pride and Prejudice,5943,128366,100,1980
1,Frankenstein,3122,75303,100,2309
2,Dracula,7839,162771,100,1731



Resumen de la preparacion de la muestra
Pride and Prejudice | oraciones totales 5,943 | tokens totales 128,366 | oraciones muestra 100 | tokens muestra 1,980
Frankenstein | oraciones totales 3,122 | tokens totales 75,303 | oraciones muestra 100 | tokens muestra 2,309
Dracula | oraciones totales 7,839 | tokens totales 162,771 | oraciones muestra 100 | tokens muestra 1,731


In [30]:
#instalamos el modelo de spacy para etiquetado gramatical

!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.8 MB 6.5 MB/s eta 0:00:02
     ------------------ --------------------- 6.0/12.8 MB 15.1 MB/s eta 0:00:01
     ------------------------------------ -- 12.1/12.8 MB 19.9 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 19.4 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
#etiquetamos cada token de las muestras usando spacy

import spacy

nlp = spacy.load("en_core_web_sm")

for titulo, datos in novelas_procesadas.items():
    muestras_pos = []

    for oracion in datos["muestra"]:
        texto_oracion = " ".join(oracion)
        doc = nlp(texto_oracion)

        etiquetas_oracion = [
            (token.text, token.pos_)
            for token in doc
        ]

        muestras_pos.append(etiquetas_oracion)

    datos["muestra_pos"] = muestras_pos

    print(f"\nLibro {titulo}")
    print("Ejemplo de oracion etiquetada")
    print(datos["muestra_pos"][0])


Libro Pride and Prejudice
Ejemplo de oracion etiquetada
[('After', 'ADP'), ('this', 'DET'), ('day', 'NOUN'), ('Jane', 'PROPN'), ('said', 'VERB'), ('no', 'DET'), ('more', 'ADJ'), ('of', 'ADP'), ('her', 'PRON'), ('indifference', 'NOUN')]

Libro Frankenstein
Ejemplo de oracion etiquetada
[('I', 'PRON'), ('was', 'AUX'), ('partly', 'ADV'), ('urged', 'VERB'), ('by', 'ADP'), ('curiosity', 'NOUN'), ('and', 'CCONJ'), ('compassion', 'NOUN'), ('confirmed', 'VERB'), ('my', 'PRON'), ('resolution', 'NOUN')]

Libro Dracula
Ejemplo de oracion etiquetada
[('I', 'PRON'), ('shall', 'AUX'), ('wire', 'VERB'), ('to', 'ADP'), ('my', 'PRON'), ('people', 'NOUN'), ('to', 'PART'), ('have', 'VERB'), ('horses', 'NOUN'), ('and', 'CCONJ'), ('carriages', 'NOUN'), ('where', 'SCONJ'), ('they', 'PRON'), ('will', 'AUX'), ('be', 'AUX'), ('most', 'ADV'), ('convenient', 'ADJ'), ('Look', 'PROPN'), ('here', 'ADV'), ('old', 'ADJ'), ('fellow', 'NOUN'), ('said', 'VERB'), ('Morris', 'PROPN'), ('it', 'PRON'), ('is', 'AUX'), ('a',

In [32]:
#calculamos la distribucion de categorias gramaticales de cada libro

from collections import Counter

distribuciones_pos = {}

for titulo, datos in novelas_procesadas.items():
    conteo_pos = Counter()

    for oracion in datos["muestra_pos"]:
        for palabra, etiqueta in oracion:
            conteo_pos[etiqueta] += 1

    distribuciones_pos[titulo] = conteo_pos

#construimos una tabla comparativa entre los tres libros

categorias_pos = sorted(
    set().union(
        *[set(conteo.keys()) for conteo in distribuciones_pos.values()]
    )
)

tabla_pos = pd.DataFrame(index=categorias_pos)

for titulo, conteo in distribuciones_pos.items():
    tabla_pos[titulo] = [
        conteo.get(categoria, 0)
        for categoria in categorias_pos
    ]

tabla_pos.index.name = "POS"
tabla_pos = tabla_pos.fillna(0).astype(int)

display(tabla_pos)

,Pride and Prejudice,Frankenstein,Dracula
POS,,,
ADJ,122,160,106
ADP,213,289,186
ADV,148,98,87
AUX,181,160,123
CCONJ,77,106,85
DET,153,233,141
INTJ,11,4,4
NOUN,262,452,254
NUM,10,12,7


In [33]:
#buscamos palabras que recibieron distintas etiquetas en diferentes contextos

ocurrencias_palabras = {}

for titulo, datos in novelas_procesadas.items():
    for oracion in datos["muestra_pos"]:
        for i, (palabra, etiqueta) in enumerate(oracion):
            palabra_clave = palabra.lower()

            if palabra_clave not in ocurrencias_palabras:
                ocurrencias_palabras[palabra_clave] = []

            inicio = max(0, i - 3)
            fin = min(len(oracion), i + 4)

            contexto = " ".join(
                token
                for token, _ in oracion[inicio:fin]
            )

            ocurrencias_palabras[palabra_clave].append({
                "libro": titulo,
                "palabra": palabra,
                "etiqueta": etiqueta,
                "contexto": contexto
            })

#seleccionamos palabras con mas de una etiqueta distinta

ambiguedades_pos = []

for palabra, ocurrencias in ocurrencias_palabras.items():
    etiquetas_distintas = set(
        ocurrencia["etiqueta"]
        for ocurrencia in ocurrencias
    )

    if len(etiquetas_distintas) > 1:
        ambiguedades_pos.append(
            (palabra, ocurrencias)
        )

#mostramos al menos tres palabras con ambiguedad de etiquetas

palabras_mostradas = 0

for palabra, ocurrencias in ambiguedades_pos:
    etiquetas_mostradas = set()

    print(f"\nPalabra {palabra}")

    for ocurrencia in ocurrencias:
        if ocurrencia["etiqueta"] not in etiquetas_mostradas:
            print(f"Libro {ocurrencia['libro']}")
            print(f"Etiqueta {ocurrencia['etiqueta']}")
            print(f"Contexto {ocurrencia['contexto']}")
            print()

            etiquetas_mostradas.add(
                ocurrencia["etiqueta"]
            )

    if len(etiquetas_mostradas) > 1:
        palabras_mostradas += 1

    if palabras_mostradas == 3:
        break


Palabra after
Libro Pride and Prejudice
Etiqueta ADP
Contexto After this day Jane

Libro Pride and Prejudice
Etiqueta SCONJ
Contexto Soon after you left me


Palabra this
Libro Pride and Prejudice
Etiqueta DET
Contexto After this day Jane said

Libro Pride and Prejudice
Etiqueta PRON
Contexto This was one point


Palabra no
Libro Pride and Prejudice
Etiqueta DET
Contexto day Jane said no more of her

Libro Pride and Prejudice
Etiqueta PRON
Contexto but could do no more

Libro Frankenstein
Etiqueta ADV
Contexto human frame could no longer support the



Algunas asignaciones parecen incorrectas, especialmente en palabras que cambian de función según el contexto, como “no” o “after”, y en casos como “Look” en Dracula, que fue etiquetado como PROPN aunque funciona como verbo. De los tres, Dracula muestra errores más visibles, probablemente por sus formas de diálogo, vocabulario y construcciones menos comunes para un tagger entrenado con inglés contemporáneo.

In [34]:
#extraemos las etiquetas universal y penn treebank de 10 tokens

titulo_libro = "Pride and Prejudice"
oracion_ejemplo = novelas_procesadas[titulo_libro]["muestra"][0]

doc = nlp(" ".join(oracion_ejemplo))

resultado_tags = []

for token in doc[:10]:
    resultado_tags.append({
        "token": token.text,
        "POS_universal": token.pos_,
        "tag_Penn_Treebank": token.tag_
    })

tabla_tags = pd.DataFrame(resultado_tags)

display(tabla_tags)

,token,POS_universal,tag_Penn_Treebank
0,After,ADP,IN
1,this,DET,DT
2,day,NOUN,NN
3,Jane,PROPN,NNP
4,said,VERB,VBD
5,no,DET,DT
6,more,ADJ,JJR
7,of,ADP,IN
8,her,PRON,PRP$
9,indifference,NOUN,NN


El tagset universal muestra la categoría gramatical general de cada palabra, mientras que Penn Treebank aporta un nivel de detalle mayor. Por ejemplo, “said” aparece como VERB, pero VBD indica específicamente que es un verbo en pasado; “more” es ADJ, pero JJR indica que es un adjetivo comparativo, y “Jane” es PROPN, mientras que NNP especifica que es un nombre propio en singular. En otros casos, como “day” (NOUN → NN) o “this” (DET → DT), la etiqueta fina también precisa el tipo concreto de palabra. Por eso, Penn Treebank permite representar diferencias gramaticales que el tagset universal agrupa en una sola categoría.

In [35]:
#seleccionamos cinco oraciones de la muestra y mostramos sus etiquetas automaticas

titulo_libro = "Pride and Prejudice"

for numero, oracion in enumerate(
    novelas_procesadas[titulo_libro]["muestra"][:5],
    start=1
):
    texto_oracion = " ".join(oracion)
    doc = nlp(texto_oracion)

    print(f"\nOracion {numero}")
    print(texto_oracion)
    
    for token in doc:
        print(f"{token.text} -> {token.tag_}")


Oracion 1
After this day Jane said no more of her indifference
After -> IN
this -> DT
day -> NN
Jane -> NNP
said -> VBD
no -> DT
more -> JJR
of -> IN
her -> PRP$
indifference -> NN

Oracion 2
Good-bye She then ran gaily off rejoicing as she rambled about in the hope of being at home again in a day or two
Good -> UH
- -> UH
bye -> UH
She -> PRP
then -> RB
ran -> VBD
gaily -> RB
off -> RB
rejoicing -> VBG
as -> IN
she -> PRP
rambled -> VBD
about -> IN
in -> IN
the -> DT
hope -> NN
of -> IN
being -> VBG
at -> IN
home -> NN
again -> RB
in -> IN
a -> DT
day -> NN
or -> CC
two -> CD

Oracion 3
Bennet how can you abuse your own children in such a way
Bennet -> NN
how -> WRB
can -> MD
you -> PRP
abuse -> VB
your -> PRP$
own -> JJ
children -> NNS
in -> IN
such -> PDT
a -> DT
way -> NN

Oracion 4
Darcy s treatment of him she tried to remember something of that gentleman s reputed disposition when quite a lad which might agree with it and was confident at last that she recollected having heard M

Los desacuerdos son principalmente errores del modelo, como Bennet -> NN en vez de NNP o quite -> PDT en vez de RB. También hay algunos casos de ambigüedad genuina, como about, off o more, que dependen del contexto. La separación de “Good-bye” parece más un problema de tokenización que de etiquetas.